# NB_LIB_TRANSFORM_SLV_GLD — Thư viện dùng chung cho luồng silver → silver L2 → gold (tính lại toàn bộ)

Tạo 05/10/2026 cho luồng `PL_VV_TRANSFORM_SLV_TO_GLD_1H` (GOLD_POI_FLOW_DESIGN §5). Không phụ thuộc POI: mọi luồng
"tính lại toàn bộ + MERGE row_hash" khác dùng lại được, chỉ cần dòng cấu hình + notebook node.

**Cách dùng** (cell riêng, ngay sau cell tham số):
```
%run NB_LIB_TRANSFORM_SLV_GLD
```

**Quy ước**
- Chỉ định nghĩa hằng số, class, hàm. Không đọc / ghi khi được `%run`, không `spark.conf.set`.
- Default lakehouse `lh_vv_bronze` (giống extract, bắt buộc với runMultiple). Tên bảng luôn đầy đủ `lakehouse.schema.table`.
- Tự chứa: không `%run NB_LIB_EXTRACT_RAWDATA` (khoá / watermark của lib extract gắn với `RunContext` của 1 nguồn raw).
  Các hàm `with_retry`, `append_rows`, `quote_name`… **chép mẫu** từ lib extract, giữ nguyên hành vi.

**Mô hình** (chi tiết §5 thiết kế)
- **Edge** = 1 dòng `ctrl_mng_pipeline_config` có `load_mode = 'RECOMPUTE'` (src = bảng đầu vào, trg = node).
  Cùng khoá (src, trg) với 1 dòng `ctrl_mng_watermark` id `wm_e__<trg_tbl>__<src_tbl>`.
- **Node** = 1 `trg_schema.trg_tbl`. Notebook = cột `nb_name` (nếu bảng config có cột này và có giá trị),
  không thì `NB_<UPPER(trg_tbl)>`. Mỗi node thuộc đúng 1 pl đang active.
- **New Edge** khi: chưa đọc lần nào; bảng đầu vào bị tạo lại (Delta table id khác); có commit **đổi dữ liệu** sau
  version đã đọc (đọc `DeltaTable.history()`: bỏ OPTIMIZE / VACUUM / đổi property…, MERGE 0 dòng); history không đủ.
- **NB_00** (1 notebook cho mọi luồng, `p_pl_name`): khoá luồng → lập kế hoạch → chốt version đầu vào ngoài →
  runMultiple → đọc kết quả từng node từ `ctrl_log_table_run` → tiến cạnh (MERGE `ctrl_mng_watermark`) → run log.
  NB_00 là nơi duy nhất ghi `ctrl_mng_watermark` / `ctrl_log_run`.
- **Node** (`run_node`): đọc `VERSION AS OF` → hộp thư rỗng thì `NO_DATA` → tính lại toàn bộ → QG → MERGE theo
  `row_hash` (+ xoá mềm khi biến mất) → append 1 dòng `ctrl_log_table_run` (append không xung đột giữa các node).

| # | Nhóm | Nội dung |
|---|---|---|
| 1 | constants | Bảng ctrl, chế độ chạy, commit không đổi dữ liệu, cột kỹ thuật |
| 2 | common | Thời gian, tham số, log, retry, tên bảng, append theo schema |
| 3 | delta | Trạng thái bảng (version, table id), history, commit đổi dữ liệu, đọc theo version |
| 4 | config / DAG | Đọc cạnh, dựng DAG, kiểm tra chu trình / 1 chủ / nhịp |
| 5 | ctrl | Hộp thư cạnh, khoá luồng, run log, table log |
| 6 | node | `NodeSpec`, `NodeContext`, QG, MERGE row_hash, `run_node` |
| 7 | orchestrator | `plan_run`, DAG runMultiple, gom kết quả, tiến cạnh, `orchestrate` |
| 8 | build helpers | Chuẩn hoá chữ, bỏ dấu, slug, uuid md5, epoch, JSON |

**Cột thêm vào bảng ctrl** (NULL được, `NB_CREATE_DDL` thêm): `ctrl_log_table_run.src_versions_json`,
`trg_version`, `deactivated_rows`, `qg_json`; `ctrl_log_run.output_versions_json`. Lib extract ghi theo schema bảng
(`append_rows`) nên không ảnh hưởng luồng extract.


In [ ]:
# =============================================================================
# 1. constants
# =============================================================================
import json
import random
import re
import threading
import time
import unicodedata
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Callable, Optional

from delta.tables import DeltaTable
from pyspark import StorageLevel
from pyspark.sql import Column, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

try:
    from pyspark.errors import AnalysisException          # Spark >= 3.4
except ImportError:                                        # pragma: no cover
    from pyspark.sql.utils import AnalysisException

CTRL_SCHEMA       = "lh_vv_bronze.ctrl"
T_PIPELINE_CONFIG = f"{CTRL_SCHEMA}.ctrl_mng_pipeline_config"
T_WATERMARK       = f"{CTRL_SCHEMA}.ctrl_mng_watermark"
T_LOG_RUN         = f"{CTRL_SCHEMA}.ctrl_log_run"
T_LOG_TABLE_RUN   = f"{CTRL_SCHEMA}.ctrl_log_table_run"

EDGE_LOAD_MODE = "RECOMPUTE"                 # load_mode của cạnh luồng tính lại (extract dùng CDC / DOC / LANG...)
RUN_MODES      = ("RUN", "PLAN", "DRY_RUN")  # NB_00: PLAN = chỉ lập kế hoạch, không ghi gì
NODE_MODES     = ("RUN", "DRY_RUN")          # node: DRY_RUN = tính + QG + đếm diff, không tạo / ghi bảng đích
NODE_OK        = ("SUCCESS", "NO_DATA")
NODE_BAD       = ("FAILED", "SKIPPED", "NOT_RUN")
WATERMARK_COLUMN_EDGE = "commit_ts"          # watermark_value của cạnh = thời điểm commit của version đã đọc

# Commit KHÔNG đổi dữ liệu (operation trong DeltaTable.history()). Tham số p_non_data_ops của NB_00 thay được danh sách.
# Không có: DROP COLUMNS, RESTORE, TRUNCATE, CREATE OR REPLACE... -> coi là đổi dữ liệu (an toàn).
DEFAULT_NON_DATA_OPS = (
    "OPTIMIZE", "VACUUM START", "VACUUM END", "SET TBLPROPERTIES", "UNSET TBLPROPERTIES", "ADD COLUMNS",
    "CHANGE COLUMN", "REPLACE COLUMNS", "UPGRADE PROTOCOL", "ADD CONSTRAINT", "DROP CONSTRAINT", "FSCK",
    "CREATE TABLE", "REORG",
)
# Số dòng đổi theo operation (operationMetrics). Thiếu hết các khoá -> coi là đổi (an toàn).
OP_ROW_METRICS = {
    "MERGE":                  ("numTargetRowsInserted", "numTargetRowsUpdated", "numTargetRowsDeleted"),
    "UPDATE":                 ("numUpdatedRows",),
    "DELETE":                 ("numDeletedRows",),
    "WRITE":                  ("numOutputRows",),
    "STREAMING UPDATE":       ("numOutputRows",),
    "CREATE TABLE AS SELECT": ("numOutputRows",),
}

# Cột kỹ thuật bảng đích của node (thêm sau cột nghiệp vụ). row_hash không gồm các cột này.
TECH_COLUMNS = (("row_hash", "string"), ("created_at", "timestamp"), ("updated_at", "timestamp"),
                ("deleted_at", "timestamp"))
TECH_NAMES = tuple(n for n, _ in TECH_COLUMNS)
TABLE_PROPERTIES = ("'delta.autoOptimize.optimizeWrite' = 'true', 'delta.autoOptimize.autoCompact' = 'true'")

LOCK_TIMEOUT_DEFAULT_MIN = 90               # node chạy tay (không qua NB_00) dùng mốc này để xét khoá quá hạn
CADENCE_RE = re.compile(r"_(\d+)([MHD])$")   # nhịp lấy từ hậu tố pl_name: _15M, _1H, _1D
CADENCE_UNIT_MIN = {"M": 1, "H": 60, "D": 1440}
IDENT_RE = re.compile(r"[A-Za-z][A-Za-z0-9_]*")
NEW_LOG_COLUMNS = {T_LOG_TABLE_RUN: ("src_versions_json", "trg_version", "deactivated_rows", "qg_json"),
                   T_LOG_RUN: ("output_versions_json",)}

## 2. common — thời gian, tham số, log, retry, tên bảng, append

In [ ]:
# =============================================================================
# 2. common
# =============================================================================
_LOG_LOCK = threading.Lock()

TRANSIENT_MARKERS = (                         # chép từ lib extract (§5)
    "ConcurrentAppendException", "ConcurrentDeleteReadException", "ConcurrentDeleteDeleteException",
    "ConcurrentTransactionException", "ConcurrentWriteException", "MetadataChangedException",
    "ProtocolChangedException", "TooManyRequestsForCapacity", "Connection reset",
)


class ConfigError(Exception):
    """Cấu hình / tham số sai (cạnh, DAG, khoá, bảng ctrl thiếu cột). Không thử lại."""


class QualityGateError(Exception):
    """Exit QG chặn: không MERGE."""


def utc_now() -> datetime:
    """Giờ UTC không kèm tzinfo (ghi cột TIMESTAMP). Ví dụ: 16:30 giờ VN -> datetime(..., 9, 30)."""
    return datetime.now(timezone.utc).replace(tzinfo=None)


def to_bool(value) -> bool:
    """'true' / '1' / 'yes' / True -> True; còn lại False. Ví dụ: to_bool("TRUE") -> True."""
    return str(value).strip().lower() in ("1", "true", "yes", "y")


def split_csv(value) -> list:
    """" a, b ,," -> ["a", "b"]."""
    return [x.strip() for x in str(value or "").split(",") if x.strip()]


def short_error(e: BaseException, limit: int = 4000) -> str:
    """"<TênLỗi>: <nội dung>" cắt còn `limit` ký tự."""
    return f"{type(e).__name__}: {e}"[:limit]


def log(msg: str, level: str = "INFO", node: Optional[str] = None) -> None:
    """In 1 dòng log có giờ UTC, mức, node; an toàn khi nhiều luồng. Ví dụ: [09:41:31][INFO][slv_poi] ok."""
    tag = f"[{datetime.now(timezone.utc):%H:%M:%S}][{level}]" + (f"[{node}]" if node else "")
    with _LOG_LOCK:
        print(f"{tag} {msg}", flush=True)


def is_transient(e: BaseException) -> bool:
    """Lỗi tạm thời (xung đột ghi Delta, capacity bận, mất kết nối) -> nên thử lại."""
    text = f"{type(e).__name__} {e}"
    return any(m in text for m in TRANSIENT_MARKERS)


def with_retry(fn: Callable, label: str, attempts: int = 6, base_s: float = 2.0):
    """Chạy fn(); lỗi tạm thời thì chờ 2s, 4s, 8s... (+0–1s ngẫu nhiên) rồi chạy lại, tối đa `attempts` lần (như lib extract)."""
    for i in range(1, attempts + 1):
        try:
            return fn()
        except Exception as e:
            if i == attempts or not is_transient(e):
                raise
            wait = base_s * 2 ** (i - 1) + random.uniform(0, 1)
            log(f"{label}: lỗi tạm thời, thử lại sau {wait:.0f}s ({i}/{attempts - 1})", "WARN")
            time.sleep(wait)


def is_identifier(name) -> bool:
    """Tên 1 phần hợp lệ: chữ cái đầu, sau đó chữ / số / _. Ví dụ: "slv_poi" -> True, "a-b" -> False."""
    return bool(name) and IDENT_RE.fullmatch(str(name)) is not None


def is_fq_name(name) -> bool:
    """Tên nhiều phần hợp lệ. Ví dụ: "lh_vv_silver.dbo.slv_poi" -> True."""
    return bool(name) and all(is_identifier(p) for p in str(name).split("."))


def quote_name(fq_name: str) -> str:
    """"lh.dbo.t" -> "`lh`.`dbo`.`t`" (ghép vào SQL; tên đã kiểm tra bằng is_fq_name)."""
    return ".".join(f"`{p}`" for p in fq_name.split("."))


def table_exists(fq: str) -> bool:
    """Bảng có tồn tại không (spark.catalog.tableExists, như lib extract)."""
    try:
        return bool(spark.catalog.tableExists(fq))
    except Exception:
        return False


def table_columns(fq: str) -> dict:
    """{tên cột lower: kiểu simpleString} của bảng."""
    return {f.name.lower(): f.dataType.simpleString() for f in spark.table(fq).schema.fields}


def append_rows(table: str, rows: list) -> None:
    """Append list[dict] vào bảng Delta theo đúng schema bảng (thứ tự + kiểu); khoá thiếu -> NULL; thử lại khi xung đột.
    Chép từ lib extract. Ví dụ: append_rows(T_LOG_TABLE_RUN, [{"exec_id": "...", "status": "SUCCESS"}])."""
    if not rows:
        return
    schema = spark.table(table).schema
    df = spark.createDataFrame([tuple(r.get(f.name) for f in schema.fields) for r in rows], schema)
    with_retry(lambda: df.write.format("delta").mode("append").saveAsTable(table), table)


def json_dumps(obj) -> str:
    """JSON gọn, giữ tiếng Việt, khoá sắp xếp (ổn định khi so sánh), datetime -> ISO."""
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, default=str)


def current_notebook_name(default: str = "") -> str:
    """Tên notebook đang chạy (notebookutils.runtime.context), không có thì `default`."""
    try:
        return notebookutils.runtime.context.get("currentNotebookName") or default   # noqa: F821
    except Exception:
        return default


def cadence_minutes(pl_name: str) -> Optional[int]:
    """Nhịp của pl theo hậu tố tên. Ví dụ: "PL_X_1H" -> 60 ; "PL_X_15M" -> 15 ; "PL_X" -> None."""
    m = CADENCE_RE.search(pl_name or "")
    return int(m.group(1)) * CADENCE_UNIT_MIN[m.group(2)] if m else None


def edge_wm_id(src_tbl: str, trg_tbl: str) -> str:
    """Id dòng watermark của cạnh. Ví dụ: ("slv_3p_poi", "slv_poi") -> "wm_e__slv_poi__slv_3p_poi"."""
    return f"wm_e__{trg_tbl}__{src_tbl}"


def flow_lock_id(pl_name: str) -> str:
    """Id dòng khoá luồng. Ví dụ: "wm_flow__PL_VV_TRANSFORM_SLV_TO_GLD_1H"."""
    return f"wm_flow__{pl_name}"


def require_ctrl_columns() -> None:
    """Bảng ctrl đã có các cột gold-log chưa (NB_CREATE_DDL thêm). Thiếu -> ConfigError (append_rows sẽ bỏ cột âm thầm)."""
    missing = [f"{t}.{c}" for t, cols in NEW_LOG_COLUMNS.items() for c in cols if c.lower() not in table_columns(t)]
    if missing:
        raise ConfigError(f"Bảng ctrl thiếu cột {missing}: chạy cell thêm cột gold-log trong NB_CREATE_DDL trước")

## 3. delta — trạng thái bảng, history, commit đổi dữ liệu, đọc theo version

| Hàm | Mục đích |
|---|---|
| `table_state(fq)` | `{version, table_id, commit_ts}` hiện tại; bảng chưa có -> None |
| `table_history(fq, n)` | n commit mới nhất: version, operation, metrics, params, timestamp |
| `is_data_change(row, non_data_ops)` | 1 commit có đổi dữ liệu không |
| `HistoryCache` | Đọc history 1 lần / bảng / lần chạy, trả các commit đổi dữ liệu trong (v_from, v_to] |
| `read_at(fq, version)` / `read_current(fq)` | Đọc bảng theo version / bản hiện tại |

In [ ]:
# =============================================================================
# 3. delta
# =============================================================================
def table_state(fq: str) -> Optional[dict]:
    """Version mới nhất, Delta table id, thời điểm commit. Bảng chưa có -> None.

    Input : fq — "lh_vv_silver.dbo.slv_3p_poi"
    Output: {"version": 12, "table_id": "65f3...", "commit_ts": datetime(...)} | None
    """
    if not table_exists(fq):
        return None
    h = DeltaTable.forName(spark, fq).history(1).select("version", "timestamp").first()
    tid = spark.sql(f"DESCRIBE DETAIL {quote_name(fq)}").select("id").first()[0]
    return {"version": int(h["version"]), "table_id": str(tid) if tid else None, "commit_ts": h["timestamp"]}


def table_history(fq: str, n: int) -> list:
    """n commit mới nhất (version giảm dần).

    Output: [{"version", "operation", "metrics" (dict), "params" (dict), "ts"}]
    """
    rows = (DeltaTable.forName(spark, fq).history(int(n))
            .select("version", "operation", "operationMetrics", "operationParameters", "timestamp").collect())
    return [{"version": int(r["version"]), "operation": r["operation"], "metrics": dict(r["operationMetrics"] or {}),
             "params": dict(r["operationParameters"] or {}), "ts": r["timestamp"]} for r in rows]


def is_data_change(commit: dict, non_data_ops) -> bool:
    """1 commit có làm đổi dữ liệu không. Không chắc -> True (chạy thừa, không bỏ sót).

    Ví dụ: OPTIMIZE -> False ; MERGE numTargetRowsInserted=0, Updated=0, Deleted=0 -> False ;
           WRITE mode Overwrite -> True ; operation NULL (Eventstream) -> True ; RESTORE -> True
    """
    op = (commit.get("operation") or "").strip().upper()
    if not op:
        return True
    if op in non_data_ops:
        return False
    if op == "WRITE" and str((commit.get("params") or {}).get("mode", "")).lower() == "overwrite":
        return True
    keys = OP_ROW_METRICS.get(op)
    if not keys:
        return True
    metrics = commit.get("metrics") or {}
    present = [k for k in keys if k in metrics]
    if not present:
        return True
    try:
        return sum(int(metrics[k] or 0) for k in present) > 0
    except (TypeError, ValueError):
        return True


class HistoryCache:
    """Đọc history mỗi bảng 1 lần / lần chạy; trả commit đổi dữ liệu trong (v_from, v_to].

    history(n) trả n commit mới nhất nên n = version hiện tại - v_from (version Delta liên tục). n > history_limit
    hoặc thiếu version trong khoảng (log đã bị dọn) -> complete = False (người gọi coi là Dirty Edge).
    """

    def __init__(self, non_data_ops, history_limit: int):
        self.non_data_ops = tuple(op.upper() for op in non_data_ops)
        self.limit = int(history_limit)
        self._rows: dict = {}                     # fq -> {version: commit}
        self._from: dict = {}                     # fq -> v_from đã phủ

    def changes(self, fq: str, v_from: int, v_to: int, v_current: int) -> tuple:
        """(danh sách version đổi dữ liệu, complete). Ví dụ: (fq, 10, 13, 13) -> ([12], True)."""
        if v_to <= v_from:
            return [], True
        need = v_current - v_from
        if need > self.limit:
            return [], False
        if fq not in self._rows or self._from[fq] > v_from:
            self._rows[fq] = {c["version"]: c for c in table_history(fq, need)}
            self._from[fq] = v_from
        rows = self._rows[fq]
        expected = range(v_from + 1, v_to + 1)
        if any(v not in rows for v in expected):
            return [], False
        return [v for v in expected if is_data_change(rows[v], self.non_data_ops)], True


def read_at(fq: str, version: int) -> DataFrame:
    """Đọc bảng tại version (time travel). Ví dụ: read_at("lh_vv_silver.dbo.slv_3p_poi", 12)."""
    return spark.sql(f"SELECT * FROM {quote_name(fq)} VERSION AS OF {int(version)}")


def read_current(fq: str) -> Optional[DataFrame]:
    """Bản hiện tại của bảng, chưa có bảng -> None."""
    return spark.table(fq) if table_exists(fq) else None

## 4. config / DAG

| Hàm | Mục đích |
|---|---|
| `load_config_rows()` | Mọi cạnh đang active (`load_mode = RECOMPUTE`) của mọi pl |
| `build_dag(rows, pl_name)` | Dựng DAG của 1 pl; kiểm tra tên, 1 node 1 chủ, chu trình, nhịp giữa các pl, tên notebook |
| `node_owner(rows, trg_fq)` | pl đang sở hữu node |

In [ ]:
# =============================================================================
# 4. config / DAG
# =============================================================================
@dataclass(frozen=True)
class Edge:
    pl_name: str
    src_schema: str
    src_tbl: str
    trg_schema: str
    trg_tbl: str
    nb_name: Optional[str] = None

    @property
    def src_fq(self) -> str:
        return f"{self.src_schema}.{self.src_tbl}"

    @property
    def trg_fq(self) -> str:
        return f"{self.trg_schema}.{self.trg_tbl}"

    @property
    def wm_id(self) -> str:
        return edge_wm_id(self.src_tbl, self.trg_tbl)


@dataclass
class Dag:
    pl_name: str
    edges: list                    # Edge của pl
    nodes: dict                    # trg_fq -> {"name": trg_tbl, "nb": notebook, "edges": [Edge]}
    order: list                    # trg_fq theo thứ tự topo
    level: dict                    # trg_fq -> tầng (0 = chỉ đọc đầu vào ngoài)
    parents: dict                  # trg_fq -> set(trg_fq node cha trong pl)
    children: dict                 # trg_fq -> set(trg_fq node con trong pl)
    external: dict                 # src_fq -> pl sở hữu (node của pl khác) hoặc None (bảng ngoài)

    def descendants(self, starts) -> set:
        """Mọi hậu duệ (không gồm chính nó) của các node trong `starts`."""
        out, stack = set(), list(starts)
        while stack:
            for c in self.children.get(stack.pop(), ()):
                if c not in out:
                    out.add(c)
                    stack.append(c)
        return out

    def node_by_name(self, name: str) -> Optional[str]:
        """trg_tbl hoặc trg_fq -> trg_fq của node."""
        for fq, n in self.nodes.items():
            if name in (fq, n["name"]):
                return fq
        return None


def load_config_rows() -> list:
    """Cạnh đang active của MỌI pl (cần để kiểm tra 1 node 1 chủ). Cột nb_name tuỳ chọn (chưa có -> NULL).

    Output: list[Edge]
    """
    cols = table_columns(T_PIPELINE_CONFIG)
    if "load_mode" not in cols:
        raise ConfigError(f"{T_PIPELINE_CONFIG} chưa có cột load_mode")
    nb = "nb_name" if "nb_name" in cols else "CAST(NULL AS STRING) AS nb_name"
    rows = spark.sql(f"""
        SELECT pl_name, src_schema, src_tbl, trg_schema, trg_tbl, {nb}
        FROM {T_PIPELINE_CONFIG}
        WHERE is_active = 1 AND upper(trim(load_mode)) = :lm
    """, args={"lm": EDGE_LOAD_MODE}).collect()
    return [Edge(r["pl_name"], r["src_schema"], r["src_tbl"], r["trg_schema"], r["trg_tbl"],
                 (r["nb_name"] or "").strip() or None) for r in rows]


def node_owner(rows: list, trg_fq: str) -> list:
    """Các pl có cạnh active ghi vào node (đúng ra chỉ 1)."""
    return sorted({e.pl_name for e in rows if e.trg_fq == trg_fq})


def build_dag(rows: list, pl_name: str) -> Dag:
    """Dựng DAG của pl từ cạnh active và kiểm tra:
    - tên schema / bảng / notebook hợp lệ; không trùng cạnh; id watermark không trùng;
    - mỗi node của pl chỉ thuộc pl này (§11 chốt 1); node trùng tên bảng khác schema -> lỗi (tên activity runMultiple);
    - không chu trình; node tự đọc chính nó -> lỗi;
    - pl nhanh hơn không được đọc node của pl chậm hơn (nhịp theo hậu tố tên pl);
    - mọi cạnh của 1 node cùng nb_name.
    Lỗi -> ConfigError liệt kê hết.

    Ví dụ: pl có 33 cạnh, 8 node -> Dag(order=[slv_poi_source_map, ...], level={...})
    """
    errs = []
    mine = [e for e in rows if e.pl_name == pl_name]
    if not mine:
        raise ConfigError(f"pl '{pl_name}' không có cạnh active (load_mode = {EDGE_LOAD_MODE}) trong {T_PIPELINE_CONFIG}")
    for e in mine:
        for part in (e.src_schema, e.trg_schema):
            if not is_fq_name(part):
                errs.append(f"schema không hợp lệ '{part}'")
        for part in (e.src_tbl, e.trg_tbl):
            if not is_identifier(part):
                errs.append(f"tên bảng không hợp lệ '{part}'")
        if e.nb_name and not is_identifier(e.nb_name):
            errs.append(f"nb_name không hợp lệ '{e.nb_name}'")
        if e.src_fq == e.trg_fq:
            errs.append(f"cạnh tự đọc chính nó: {e.trg_fq}")
    seen, wm_ids = set(), {}
    for e in mine:
        k = (e.src_fq, e.trg_fq)
        if k in seen:
            errs.append(f"cạnh trùng {k}")
        seen.add(k)
        if wm_ids.get(e.wm_id, k) != k:
            errs.append(f"id watermark trùng '{e.wm_id}' cho {wm_ids[e.wm_id]} và {k}")
        wm_ids[e.wm_id] = k

    owners = {}
    for e in rows:
        owners.setdefault(e.trg_fq, set()).add(e.pl_name)
    nodes = {}
    for e in mine:
        n = nodes.setdefault(e.trg_fq, {"name": e.trg_tbl, "nb": None, "edges": [], "nb_set": set()})
        n["edges"].append(e)
        n["nb_set"].add(e.nb_name)
    for fq, n in nodes.items():
        if len(owners[fq]) > 1:
            errs.append(f"node {fq} thuộc nhiều pl đang active: {sorted(owners[fq])}")
        nbs = {x for x in n.pop("nb_set") if x}
        if len(nbs) > 1:
            errs.append(f"node {fq} có nhiều nb_name: {sorted(nbs)}")
        n["nb"] = next(iter(nbs)) if nbs else f"NB_{n['name'].upper()}"
    names = {}
    for fq, n in nodes.items():
        if n["name"] in names:
            errs.append(f"2 node trùng tên bảng '{n['name']}': {names[n['name']]}, {fq}")
        names[n["name"]] = fq

    parents = {fq: set() for fq in nodes}
    children = {fq: set() for fq in nodes}
    external = {}
    for e in mine:
        if e.src_fq in nodes:
            parents[e.trg_fq].add(e.src_fq)
            children[e.src_fq].add(e.trg_fq)
        else:
            other = sorted(owners.get(e.src_fq, set()))
            external[e.src_fq] = other[0] if other else None
    my_cad = cadence_minutes(pl_name)
    for src, other_pl in external.items():
        if other_pl is None:
            continue
        o_cad = cadence_minutes(other_pl)
        if my_cad is None or o_cad is None:
            log(f"Không đọc được nhịp của {pl_name} / {other_pl} từ hậu tố tên, bỏ qua kiểm tra nhịp cho {src}", "WARN")
        elif o_cad > my_cad:
            errs.append(f"{pl_name} ({my_cad} phút) đọc node {src} của luồng chậm hơn {other_pl} ({o_cad} phút)")

    # Kahn: thứ tự topo + tầng; còn node chưa xếp -> chu trình
    indeg = {fq: len(parents[fq]) for fq in nodes}
    level = {fq: 0 for fq in nodes}
    queue = sorted(fq for fq, d in indeg.items() if d == 0)
    order = []
    while queue:
        fq = queue.pop(0)
        order.append(fq)
        for c in sorted(children[fq]):
            level[c] = max(level[c], level[fq] + 1)
            indeg[c] -= 1
            if indeg[c] == 0:
                queue.append(c)
    if len(order) != len(nodes):
        errs.append(f"DAG có chu trình: {sorted(set(nodes) - set(order))}")
    if errs:
        raise ConfigError("Cấu hình DAG sai:\n  - " + "\n  - ".join(errs))
    return Dag(pl_name, mine, nodes, order, level, parents, children, external)

## 5. ctrl — hộp thư cạnh, khoá luồng, run log, table log

| Hàm | Mục đích |
|---|---|
| `load_edge_wm(edges)` | Dòng watermark của các cạnh: version / table id đã đọc |
| `edge_status(...)` | Dirty Edge không + lý do |
| `merge_edge_wm(rows)` | Tiến cạnh / ghi trạng thái (1 MERGE, tự tạo dòng còn thiếu). Chỉ NB_00 gọi |
| `ensure_lock_row`, `acquire_flow_lock`, `release_flow_lock`, `read_flow_lock` | Khoá cả luồng trên dòng `wm_flow__<pl>` |
| `close_stale_runs`, `write_run_log`, `update_run_log` | Run log của NB_00 |
| `write_node_log`, `node_logs(exec_id)` | Table log của node (append) |

In [ ]:
# =============================================================================
# 5. ctrl
# =============================================================================
WM_EDGE_SCHEMA = T.StructType([
    T.StructField("watermark_id", T.StringType()), T.StructField("src_schema", T.StringType()),
    T.StructField("src_tbl", T.StringType()), T.StructField("trg_schema", T.StringType()),
    T.StructField("trg_tbl", T.StringType()), T.StructField("advance", T.BooleanType()),
    T.StructField("last_src_version", T.LongType()), T.StructField("last_src_table_id", T.StringType()),
    T.StructField("watermark_value", T.TimestampType()), T.StructField("status", T.StringType()),
    T.StructField("flow_name", T.StringType()), T.StructField("last_run_id", T.StringType()),
    T.StructField("error_message", T.StringType()),
])


def load_edge_wm(edges: list) -> dict:
    """Dòng watermark của các cạnh, khoá (src_fq, trg_fq). Cạnh chưa có dòng -> không có trong dict (= chưa đọc lần nào).
    Dòng có id đúng nhưng src / trg lệch config -> ConfigError (tránh tiến nhầm cạnh).

    Output: {(src_fq, trg_fq): {"version": 12 | None, "table_id": ..., "ts": ..., "status": ...}}
    """
    ids = {e.wm_id: e for e in edges}
    if not ids:
        return {}
    rows = (spark.table(T_WATERMARK).where(F.col("watermark_id").isin(list(ids)))
            .select("watermark_id", "src_schema", "src_tbl", "trg_schema", "trg_tbl", "last_src_version",
                    "last_src_table_id", "watermark_value", "status").collect())
    out, errs, dup = {}, [], set()
    for r in rows:
        e = ids[r["watermark_id"]]
        k = (e.src_fq, e.trg_fq)
        if (r["src_schema"], r["src_tbl"], r["trg_schema"], r["trg_tbl"]) != (e.src_schema, e.src_tbl, e.trg_schema, e.trg_tbl):
            errs.append(f"{r['watermark_id']}: dòng watermark ({r['src_schema']}.{r['src_tbl']} -> "
                        f"{r['trg_schema']}.{r['trg_tbl']}) lệch config {k}")
        if k in out:
            dup.add(r["watermark_id"])
        v = r["last_src_version"]
        out[k] = {"version": None if v is None else int(v), "table_id": r["last_src_table_id"],
                  "ts": r["watermark_value"], "status": r["status"]}
    if dup:
        errs.append(f"trùng dòng watermark: {sorted(dup)}")
    if errs:
        raise ConfigError("ctrl_mng_watermark sai:\n  - " + "\n  - ".join(errs))
    return out


def edge_status(consumed: Optional[dict], state: Optional[dict], read_version: Optional[int],
                hist: HistoryCache, src_fq: str) -> tuple:
    """Dirty Edge không.

    Input : consumed — dòng watermark của cạnh (load_edge_wm) hoặc None ; state — table_state(src) ;
            read_version — version sẽ đọc (đã chốt / hiện tại) ; hist — HistoryCache
    Output: (dirty: bool, reason: str, versions: list)
    Ví dụ : chưa đọc lần nào -> (True, "NEW_EDGE", []) ; table id khác -> (True, "TABLE_RECREATED", [])
            có MERGE 3 dòng ở v13 sau v12 -> (True, "DATA_CHANGED", [13]) ; chỉ OPTIMIZE -> (False, "MAINTENANCE_ONLY", [])
    """
    if state is None or read_version is None:
        return True, "MISSING_TABLE", []
    if not consumed or consumed.get("version") is None:
        return True, "NEW_EDGE", []
    if consumed.get("table_id") and state.get("table_id") and consumed["table_id"] != state["table_id"]:
        return True, "TABLE_RECREATED", []
    cv = consumed["version"]
    if read_version < cv:
        return True, "VERSION_BACKWARD", []
    if read_version == cv:
        return False, "UNCHANGED", []
    changed, complete = hist.changes(src_fq, cv, read_version, max(read_version, state["version"]))
    if not complete:
        return True, "HISTORY_GAP", []
    if changed:
        return True, "DATA_CHANGED", changed[:20]
    return False, "MAINTENANCE_ONLY", []


def sql_merge_edge_wm(view: str) -> str:
    """Câu MERGE tiến cạnh (tách hàm để test parse). advance = true: ghi version / table id / mốc commit, chỉ tiến không lùi
    (trừ khi bảng bị tạo lại); advance = false: chỉ ghi trạng thái + lỗi. Cạnh chưa có dòng -> INSERT."""
    return f"""
MERGE INTO {quote_name(T_WATERMARK)} AS t
USING {view} AS s
ON t.watermark_id = s.watermark_id
WHEN MATCHED AND s.advance AND (t.last_src_version IS NULL OR t.last_src_table_id IS NULL
                                OR t.last_src_table_id <> s.last_src_table_id
                                OR s.last_src_version >= t.last_src_version) THEN UPDATE SET
    t.last_src_version  = s.last_src_version,
    t.last_src_table_id = s.last_src_table_id,
    t.watermark_column  = '{WATERMARK_COLUMN_EDGE}',
    t.watermark_value   = s.watermark_value,
    t.last_success_at   = current_timestamp(),
    t.status            = s.status,
    t.flow_name         = s.flow_name,
    t.last_run_id       = s.last_run_id,
    t.error_message     = NULL,
    t.updated_at        = current_timestamp()
WHEN MATCHED AND NOT s.advance THEN UPDATE SET
    t.status        = s.status,
    t.last_run_id   = s.last_run_id,
    t.error_message = s.error_message,
    t.updated_at    = current_timestamp()
WHEN NOT MATCHED THEN INSERT
    (watermark_id, src_schema, src_tbl, trg_schema, trg_tbl, watermark_column, watermark_value, last_success_at,
     status, flow_name, watermark_sequence, last_run_id, last_src_version, last_src_table_id, error_message,
     lock_exec_id, lock_at, updated_at)
VALUES
    (s.watermark_id, s.src_schema, s.src_tbl, s.trg_schema, s.trg_tbl, '{WATERMARK_COLUMN_EDGE}',
     CASE WHEN s.advance THEN s.watermark_value END, CASE WHEN s.advance THEN current_timestamp() END,
     s.status, s.flow_name, CAST(NULL AS BIGINT), s.last_run_id,
     CASE WHEN s.advance THEN s.last_src_version END, CASE WHEN s.advance THEN s.last_src_table_id END,
     s.error_message, CAST(NULL AS STRING), CAST(NULL AS TIMESTAMP), current_timestamp())
"""


def merge_edge_wm(rows: list, tag: str) -> None:
    """Ghi các dòng cạnh bằng 1 MERGE (thử lại khi xung đột). rows: dict theo WM_EDGE_SCHEMA."""
    if not rows:
        return
    df = spark.createDataFrame([tuple(r.get(f.name) for f in WM_EDGE_SCHEMA.fields) for r in rows], WM_EDGE_SCHEMA)
    view = f"_v_wm_{tag}"
    df.createOrReplaceTempView(view)
    try:
        with_retry(lambda: spark.sql(sql_merge_edge_wm(view)), "ctrl_mng_watermark")
    finally:
        spark.catalog.dropTempView(view)


def sql_ensure_lock_row(view: str) -> str:
    """MERGE chỉ INSERT dòng khoá nếu chưa có (2 lần tạo cùng lúc: 1 lần bị xung đột, thử lại thì đã có dòng)."""
    return f"""
MERGE INTO {quote_name(T_WATERMARK)} AS t
USING {view} AS s
ON t.watermark_id = s.watermark_id
WHEN NOT MATCHED THEN INSERT
    (watermark_id, src_schema, src_tbl, trg_schema, trg_tbl, watermark_column, watermark_value, last_success_at,
     status, flow_name, watermark_sequence, last_run_id, last_src_version, last_src_table_id, error_message,
     lock_exec_id, lock_at, updated_at)
VALUES
    (s.watermark_id, CAST(NULL AS STRING), s.src_tbl, CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING),
     CAST(NULL AS TIMESTAMP), CAST(NULL AS TIMESTAMP), 'INITIALIZED', s.flow_name, CAST(NULL AS BIGINT),
     CAST(NULL AS STRING), CAST(NULL AS BIGINT), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING),
     CAST(NULL AS TIMESTAMP), current_timestamp())
"""


def ensure_lock_row(pl_name: str, flow_name: str) -> str:
    """Tạo dòng khoá wm_flow__<pl> nếu chưa có (src_tbl = pl_name, trg NULL). Output: watermark_id."""
    wid = flow_lock_id(pl_name)
    df = spark.createDataFrame([(wid, pl_name, flow_name)], "watermark_id string, src_tbl string, flow_name string")
    view = f"_v_lock_{uuid.uuid4().hex[:8]}"
    df.createOrReplaceTempView(view)
    try:
        with_retry(lambda: spark.sql(sql_ensure_lock_row(view)), "ctrl_mng_watermark")
    finally:
        spark.catalog.dropTempView(view)
    return wid


def _lock_expired_sql(timeout_min: int) -> str:
    """Khoá quá hạn (lock_at NULL coi như quá hạn), tính bằng current_timestamp() như lúc ghi lock_at."""
    return f"coalesce(lock_at < current_timestamp() - INTERVAL {int(timeout_min)} MINUTES, true)"


def read_flow_lock(pl_name: str, timeout_min: int):
    """Row(lock_exec_id, lock_at, expired) của dòng khoá; chưa có dòng -> None."""
    return spark.sql(f"""
        SELECT lock_exec_id, lock_at, {_lock_expired_sql(timeout_min)} AS expired
        FROM {T_WATERMARK} WHERE watermark_id = :wid
    """, args={"wid": flow_lock_id(pl_name)}).first()


def _update_lock_row(pl_name: str, values: dict, condition: Column) -> None:
    cond = (F.col("watermark_id") == F.lit(flow_lock_id(pl_name))) & condition
    with_retry(lambda: DeltaTable.forName(spark, T_WATERMARK).update(
        cond, {**values, "updated_at": F.current_timestamp()}), "ctrl_mng_watermark")


def acquire_flow_lock(pl_name: str, exec_id: str, run_id: str, timeout_min: int) -> tuple:
    """Nhận khoá luồng (nguyên tử nhờ Delta optimistic concurrency, cùng cách lib extract):
    UPDATE ... SET lock_exec_id = exec_id WHERE (lock_exec_id IS NULL OR quá hạn), rồi đọc lại.

    Output: (True, None) | (False, "exec_id ... đang giữ khoá từ ...")
    """
    before = read_flow_lock(pl_name, timeout_min)
    if before is None:
        raise ConfigError(f"Chưa có dòng khoá {flow_lock_id(pl_name)} (ensure_lock_row)")
    if before["lock_exec_id"] and not before["expired"]:
        return False, f"exec_id {before['lock_exec_id']} đang giữ khoá từ {before['lock_at']}"
    free = F.col("lock_exec_id").isNull() | F.expr(_lock_expired_sql(timeout_min))
    _update_lock_row(pl_name, {"lock_exec_id": F.lit(exec_id), "lock_at": F.current_timestamp(),
                               "status": F.lit("RUNNING"), "last_run_id": F.lit(run_id)}, free)
    after = read_flow_lock(pl_name, timeout_min)
    if after["lock_exec_id"] != exec_id:
        return False, f"exec_id {after['lock_exec_id']} nhận khoá trước (từ {after['lock_at']})"
    if before["lock_exec_id"]:
        log(f"Lấy lại khoá quá hạn của exec_id {before['lock_exec_id']} (giữ từ {before['lock_at']})", "WARN")
    return True, None


def release_flow_lock(pl_name: str, exec_id: str, status: str) -> None:
    """Nhả khoá nếu run này còn giữ (khoá đã bị run khác lấy lại vì quá hạn -> không sửa gì)."""
    _update_lock_row(pl_name, {"lock_exec_id": F.lit(None).cast("string"), "lock_at": F.lit(None).cast("timestamp"),
                               "status": F.lit(status)}, F.col("lock_exec_id") == F.lit(exec_id))


def close_stale_runs(pl_name: str, exec_id: str, timeout_min: int) -> list:
    """Run log RUNNING của pl (khác run này) quá hạn khoá -> ABANDONED (NB_00 chết không đóng log). Output: exec_id đã đóng."""
    rows = spark.sql(f"""
        SELECT exec_id FROM {T_LOG_RUN}
        WHERE pl_name = :pl AND status = 'RUNNING' AND exec_id <> :me
          AND coalesce(started_at < current_timestamp() - INTERVAL {int(timeout_min)} MINUTES, true)
    """, args={"pl": pl_name, "me": exec_id}).collect()
    stale = [r["exec_id"] for r in rows]
    if stale:
        with_retry(lambda: DeltaTable.forName(spark, T_LOG_RUN).update(
            F.col("exec_id").isin(stale),
            {"status": F.lit("ABANDONED"), "ended_at": F.current_timestamp(),
             "error_message": F.lit(f"RUNNING quá {timeout_min} phút, coi như đã dừng")}), "ctrl_log_run")
        log(f"Đánh dấu ABANDONED: {stale}", "WARN")
    return stale


def write_run_log(row: dict) -> None:
    """Append 1 dòng ctrl_log_run (RUNNING hoặc dòng cuối khi không giữ được khoá)."""
    append_rows(T_LOG_RUN, [row])


def update_run_log(exec_id: str, values: dict) -> None:
    """Cập nhật dòng run log của exec_id (đóng run). values: {cột: giá trị Python}."""
    cols = table_columns(T_LOG_RUN)
    sets = {k: F.lit(v).cast(cols[k.lower()]) for k, v in values.items() if k.lower() in cols}
    with_retry(lambda: DeltaTable.forName(spark, T_LOG_RUN).update(F.col("exec_id") == F.lit(exec_id), sets),
               "ctrl_log_run")


def write_node_log(row: dict) -> None:
    """Append 1 dòng ctrl_log_table_run của node (blind append: nhiều node cùng lúc không xung đột)."""
    append_rows(T_LOG_TABLE_RUN, [row])


def node_logs(exec_id: str) -> dict:
    """Dòng log mới nhất của từng node trong exec. Output: {trg_fq: dict dòng log}."""
    rows = spark.table(T_LOG_TABLE_RUN).where(F.col("exec_id") == exec_id).collect()
    out = {}
    for r in sorted(rows, key=lambda x: (x["ended_at"] or datetime.min)):
        d = r.asDict()
        out[f"{d['trg_schema']}.{d['trg_tbl']}"] = d
    return out


def last_output_versions(pl_name: str) -> dict:
    """§11 chốt 2: version các bảng pl sở hữu ở run SUCCESS / NO_DATA gần nhất (lát cắt nhất quán). Không có -> {}."""
    r = spark.sql(f"""
        SELECT output_versions_json FROM {T_LOG_RUN}
        WHERE pl_name = :pl AND status IN ('SUCCESS', 'NO_DATA') AND output_versions_json IS NOT NULL
        ORDER BY ended_at DESC LIMIT 1
    """, args={"pl": pl_name}).first()
    return json.loads(r[0]) if r and r[0] else {}

## 6. node — `NodeSpec`, `NodeContext`, QG, MERGE row_hash, `run_node`

Notebook node chỉ khai báo `NodeSpec` + hàm `build(ctx) -> DataFrame` (cột nghiệp vụ, 1 dòng / khoá) rồi gọi `run_node`.

| Bước | Việc |
|---|---|
| 1 | Đọc cạnh của node trong config (đúng 1 pl sở hữu); đối chiếu `spec.inputs` với cạnh (lệch -> ConfigError) |
| 2 | Khoá: chạy qua NB_00 thì `p_exec_id` phải đang giữ khoá luồng; chạy tay RUN thì tự nhận khoá; DRY_RUN chỉ cảnh báo |
| 3 | Chạy qua NB_00: node cha lỗi / bị bỏ qua trong cùng exec -> `SKIPPED` |
| 4 | Version đọc: bảng ngoài = `p_pinned_json`, node cha = version hiện tại; hộp thư rỗng (và bảng đích đã có) -> `NO_DATA` |
| 5 | `build(ctx)` -> thêm `row_hash` -> exit QG (chặn -> `FAILED`, không ghi) -> đếm diff |
| 6 | RUN: tạo / bổ sung cột bảng đích -> MERGE (bỏ qua khi diff = 0) ; DRY_RUN: dừng ở đếm diff |
| 7 | Append 1 dòng `ctrl_log_table_run` (kể cả FAILED / SKIPPED / NO_DATA); `FAILED` / `SKIPPED` -> raise để runMultiple ghi nhận |

MERGE (cột kỹ thuật `row_hash`, `created_at`, `updated_at`, `deleted_at`):
- khớp khoá và (hash khác hoặc đang xoá mềm) -> UPDATE, `deleted_at = NULL`;
- không có ở đích -> INSERT;
- có ở đích, không còn ở nguồn, chưa xoá -> `deleted_at = now` + `spec.soft_delete_set` (vd `is_active = false`).

`row_hash = sha2(to_json(struct(cột nghiệp vụ theo thứ tự DataFrame trừ hash_exclude)), 256)`; timestamp băm bằng
`unix_micros` (không phụ thuộc timezone session). Mảng phải được `array_sort` trong `build` (lib không tự sắp).
Node khác đọc bảng của node (`ctx.input`) mặc định chỉ lấy dòng `deleted_at IS NULL` và bỏ 4 cột kỹ thuật.


In [ ]:
# =============================================================================
# 6. node
# =============================================================================
@dataclass
class NodeSpec:
    trg_tbl: str                                   # bảng đích (= tên node, = trg_tbl trong config)
    inputs: dict                                   # vai trò -> src_tbl, phải khớp đúng các cạnh của node trong config
    pk: tuple                                      # khoá MERGE
    build: Callable                                # build(ctx) -> DataFrame cột nghiệp vụ
    required_inputs: tuple = ()                    # vai trò phải có >= 1 dòng (rỗng -> FAILED, chống xoá mềm hàng loạt)
    unique_checks: tuple = ()                      # ((tên, (cột...), điều kiện lọc SQL | None), ...) -> trùng thì chặn
    block_checks: tuple = ()                       # ((tên, điều kiện SQL của dòng lỗi), ...) -> > 0 dòng thì chặn
    warn_checks: tuple = ()                        # ((tên, điều kiện SQL), ...) -> chỉ đếm, ghi qg_json
    hash_exclude: tuple = ()                       # cột không đưa vào row_hash
    soft_delete_set: dict = field(default_factory=dict)   # cột -> biểu thức SQL khi xoá mềm, vd {"is_active": "false"}
    reads_self: bool = False                       # build đọc bản hiện tại của bảng đích (trạng thái có chủ đích, vd slug)


class NodeContext:
    """Thứ `build(ctx)` dùng: đọc đầu vào theo version đã chốt, đọc bảng đích hiện tại, ghi cảnh báo / số liệu."""

    def __init__(self, spec: NodeSpec, trg_fq: str, edges_by_role: dict, read_versions: dict, exec_id: str, mode: str):
        self.spec, self.trg_fq, self.exec_id, self.mode = spec, trg_fq, exec_id, mode
        self.edges_by_role, self.read_versions = edges_by_role, read_versions
        self.warns: dict = {}
        self.metrics: dict = {}

    def input(self, role: str, live_only: bool = True) -> DataFrame:
        """DataFrame của đầu vào theo version đã chốt. Bảng do lib này ghi (có row_hash + deleted_at): live_only = True
        -> chỉ dòng deleted_at IS NULL và bỏ 4 cột kỹ thuật. Ví dụ: ctx.input("src_map")."""
        if role not in self.edges_by_role:
            raise ConfigError(f"{self.spec.trg_tbl}: vai trò đầu vào '{role}' không có trong spec.inputs")
        e = self.edges_by_role[role]
        df = read_at(e.src_fq, self.read_versions[e.src_fq])
        cols = {c.lower() for c in df.columns}
        if live_only and {"row_hash", "deleted_at"} <= cols:
            df = df.where(F.col("deleted_at").isNull()).drop(*[c for c in df.columns if c.lower() in TECH_NAMES])
        return df

    def is_empty(self, role: str) -> bool:
        return self.input(role).limit(1).count() == 0

    def current_target(self) -> Optional[DataFrame]:
        """Bản hiện tại của bảng đích (gồm dòng đã xoá mềm và cột kỹ thuật); chưa có bảng -> None."""
        return read_current(self.trg_fq)

    def warn(self, name: str, n) -> None:
        """Ghi 1 cảnh báo (số dòng) vào qg_json; > 0 thì in WARN."""
        self.warns[name] = int(n or 0)
        if self.warns[name]:
            log(f"QG cảnh báo {name}: {self.warns[name]}", "WARN", self.spec.trg_tbl)


def _hash_expr(df: DataFrame, cols: list) -> Column:
    """Biểu thức row_hash. Timestamp -> unix_micros, date -> chuỗi (không phụ thuộc timezone session)."""
    parts = []
    for c in cols:
        dt = df.schema[c].dataType
        if isinstance(dt, T.TimestampType):
            parts.append(F.expr(f"unix_micros(`{c}`)").alias(c))
        elif isinstance(dt, T.DateType):
            parts.append(F.col(c).cast("string").alias(c))
        else:
            parts.append(F.col(c).alias(c))
    return F.sha2(F.to_json(F.struct(*parts)), 256)


def add_row_hash(df: DataFrame, spec: NodeSpec) -> DataFrame:
    """Kiểm tra cột rồi thêm row_hash. DataFrame build có cột kỹ thuật / thiếu khoá / kiểu void -> ConfigError."""
    names = [c.lower() for c in df.columns]
    errs = [f"cột kỹ thuật '{c}' do lib thêm, build không được trả" for c in df.columns if c.lower() in TECH_NAMES]
    errs += [f"thiếu cột khoá '{c}'" for c in spec.pk if c.lower() not in names]
    errs += [f"cột '{f.name}' kiểu void (NULL chưa cast)" for f in df.schema.fields if isinstance(f.dataType, T.NullType)]
    errs += [f"hash_exclude có cột khoá '{c}'" for c in spec.hash_exclude if c in spec.pk]
    if len(set(names)) != len(names):
        errs.append(f"trùng tên cột {sorted({c for c in names if names.count(c) > 1})}")
    if errs:
        raise ConfigError(f"{spec.trg_tbl}: " + "; ".join(errs))
    cols = [c for c in df.columns if c not in spec.hash_exclude]
    return df.withColumn("row_hash", _hash_expr(df, cols))


def quality_gate(spec: NodeSpec, df: DataFrame, ctx: NodeContext) -> dict:
    """Exit QG trên kết quả build (đã có row_hash). Chặn: khoá NULL, khoá trùng, unique_checks, block_checks.
    Cảnh báo: warn_checks + ctx.warns. Chặn -> QualityGateError (thông điệp JSON số dòng lỗi).

    Output: {"rows": n, "blocks": {...}, "warns": {...}}
    """
    pk = list(spec.pk)
    exprs = [F.count(F.lit(1)).alias("_rows"),
             F.sum(F.when(F.expr(" OR ".join(f"`{c}` IS NULL" for c in pk)), 1).otherwise(0)).alias("_pk_null")]
    for i, (_, cond) in enumerate(spec.block_checks):
        exprs.append(F.sum(F.when(F.expr(cond), 1).otherwise(0)).alias(f"_b{i}"))
    for i, (_, cond) in enumerate(spec.warn_checks):
        exprs.append(F.sum(F.when(F.expr(cond), 1).otherwise(0)).alias(f"_w{i}"))
    agg = df.agg(*exprs).first().asDict()
    blocks = {"pk_null": int(agg["_pk_null"] or 0),
              "pk_duplicate": df.groupBy(*pk).count().where("count > 1").count()}
    for i, (name, _) in enumerate(spec.block_checks):
        blocks[name] = int(agg[f"_b{i}"] or 0)
    for name, cols, where in spec.unique_checks:
        d = df.where(where) if where else df
        d = d.where(" AND ".join(f"`{c}` IS NOT NULL" for c in cols))
        blocks[name] = d.groupBy(*cols).count().where("count > 1").count()
    for i, (name, _) in enumerate(spec.warn_checks):
        ctx.warn(name, agg[f"_w{i}"])
    qg = {"rows": int(agg["_rows"]), "blocks": blocks, "warns": dict(ctx.warns)}
    failed = {k: v for k, v in blocks.items() if v}
    if failed:
        raise QualityGateError(f"QG chặn {spec.trg_tbl}: {json_dumps(failed)}")
    return qg


def diff_counts(src: DataFrame, trg_fq: str, pk: tuple) -> dict:
    """Số dòng MERGE sẽ insert / update / xoá mềm (1 lần full outer join theo khoá). Dùng cho DRY_RUN, log, và bỏ qua
    MERGE khi không có gì đổi (không tạo commit -> node con thấy hộp thư rỗng).

    Output: {"target_exists": bool, "inserted": n, "updated": n, "deactivated": n}
    """
    t = read_current(trg_fq)
    if t is None:
        return {"target_exists": False, "inserted": src.count(), "updated": 0, "deactivated": 0}
    pk = list(pk)
    s = src.select(*pk, F.col("row_hash").alias("_s_hash"), F.lit(1).alias("_s_in"))
    tt = t.select(*pk, F.col("row_hash").alias("_t_hash"), F.col("deleted_at").alias("_t_del"), F.lit(1).alias("_t_in"))
    j = s.join(tt, pk, "full_outer")
    both = F.col("_s_in").isNotNull() & F.col("_t_in").isNotNull()
    r = j.agg(
        F.sum(F.when(F.col("_s_in").isNotNull() & F.col("_t_in").isNull(), 1).otherwise(0)).alias("ins"),
        F.sum(F.when(both & (F.col("_t_hash").isNull() | (F.col("_t_hash") != F.col("_s_hash"))
                             | F.col("_t_del").isNotNull()), 1).otherwise(0)).alias("upd"),
        F.sum(F.when(F.col("_s_in").isNull() & F.col("_t_in").isNotNull() & F.col("_t_del").isNull(), 1)
              .otherwise(0)).alias("dea"),
    ).first()
    return {"target_exists": True, "inserted": int(r["ins"] or 0), "updated": int(r["upd"] or 0),
            "deactivated": int(r["dea"] or 0)}


def target_ddl(fq: str, schema: T.StructType) -> str:
    """CREATE TABLE bảng đích từ schema build (bỏ row_hash) + 4 cột kỹ thuật; optimizeWrite + autoCompact như extract."""
    cols = [(f.name, f.dataType.simpleString()) for f in schema.fields if f.name != "row_hash"] + list(TECH_COLUMNS)
    body = ",\n    ".join(f"`{n}` {t}" for n, t in cols)
    return f"CREATE TABLE IF NOT EXISTS {quote_name(fq)} (\n    {body}\n) USING DELTA\nTBLPROPERTIES ({TABLE_PROPERTIES})"


def ensure_target(fq: str, schema: T.StructType, node: str) -> Optional[str]:
    """Chưa có bảng -> CREATE ; có cột mới -> ALTER TABLE ADD COLUMNS ; lệch kiểu -> ConfigError ;
    bảng có cột mà build không trả -> WARN (cột đó giữ giá trị cũ / NULL khi insert).

    Output: "CREATED" | "ADDED [...]" | None
    """
    wanted = [(f.name, f.dataType.simpleString()) for f in schema.fields if f.name != "row_hash"] + list(TECH_COLUMNS)
    if not table_exists(fq):
        spark.sql(target_ddl(fq, schema))
        return "CREATED"
    existing = table_columns(fq)
    mism = [f"{n} (bảng {existing[n.lower()]}, build {t})" for n, t in wanted
            if n.lower() in existing and existing[n.lower()] != t]
    if mism:
        raise ConfigError(f"{fq} lệch kiểu cột: {'; '.join(mism)}")
    extra = sorted(set(existing) - {n.lower() for n, _ in wanted})
    if extra:
        log(f"Bảng đích có cột build không trả: {extra}", "WARN", node)
    missing = [(n, t) for n, t in wanted if n.lower() not in existing]
    if missing:
        spark.sql(f"ALTER TABLE {quote_name(fq)} ADD COLUMNS (" + ", ".join(f"`{n}` {t}" for n, t in missing) + ")")
        return f"ADDED {[n for n, _ in missing]}"
    return None


def sql_merge_target(fq: str, view: str, cols: list, pk: tuple, soft_delete_set: dict) -> str:
    """Câu MERGE row_hash + xoá mềm (tách hàm để test parse).

    Input : cols — cột nghiệp vụ (không gồm row_hash) ; soft_delete_set — {cột: biểu thức SQL}
    """
    on = " AND ".join(f"t.`{c}` = s.`{c}`" for c in pk)
    upd = [f"t.`{c}` = s.`{c}`" for c in cols if c not in pk]
    upd += ["t.`row_hash` = s.`row_hash`", "t.`updated_at` = current_timestamp()", "t.`deleted_at` = NULL"]
    ins_cols = [f"`{c}`" for c in cols] + ["`row_hash`", "`created_at`", "`updated_at`", "`deleted_at`"]
    ins_vals = [f"s.`{c}`" for c in cols] + ["s.`row_hash`", "current_timestamp()", "current_timestamp()",
                                             "CAST(NULL AS TIMESTAMP)"]
    sd = ["t.`deleted_at` = current_timestamp()", "t.`updated_at` = current_timestamp()"]
    sd += [f"t.`{c}` = {expr}" for c, expr in soft_delete_set.items()]
    nl = ",\n    "
    return f"""
MERGE INTO {quote_name(fq)} AS t
USING {view} AS s
ON {on}
WHEN MATCHED AND (t.`row_hash` IS NULL OR t.`row_hash` <> s.`row_hash` OR t.`deleted_at` IS NOT NULL) THEN UPDATE SET
    {nl.join(upd)}
WHEN NOT MATCHED THEN INSERT ({', '.join(ins_cols)})
VALUES ({', '.join(ins_vals)})
WHEN NOT MATCHED BY SOURCE AND t.`deleted_at` IS NULL THEN UPDATE SET
    {nl.join(sd)}
"""


def merge_target(fq: str, df: DataFrame, spec: NodeSpec, tag: str) -> None:
    """MERGE df (đã có row_hash) vào bảng đích qua temp view tên duy nhất (các node chạy chung Spark session)."""
    view = f"_v_{spec.trg_tbl}_{tag}"
    df.createOrReplaceTempView(view)
    cols = [c for c in df.columns if c != "row_hash"]
    sql = sql_merge_target(fq, view, cols, spec.pk, spec.soft_delete_set)
    try:
        with_retry(lambda: spark.sql(sql), fq)
    finally:
        spark.catalog.dropTempView(view)


def _resolve_node(spec: NodeSpec, rows: list, p_pl_name: str, p_trg_fq: str) -> tuple:
    """Tìm node trong config, pl sở hữu, DAG, cạnh theo vai trò; đối chiếu spec.inputs với cạnh."""
    trg = sorted({e.trg_fq for e in rows if e.trg_tbl == spec.trg_tbl})
    if len(trg) != 1:
        raise ConfigError(f"{spec.trg_tbl}: cần đúng 1 bảng đích active trong config, thấy {trg or 'không có'}")
    trg_fq = trg[0]
    if p_trg_fq and p_trg_fq != trg_fq:
        raise ConfigError(f"{spec.trg_tbl}: NB_00 gọi với p_trg_fq = {p_trg_fq}, config là {trg_fq} (sai nb_name?)")
    owners = node_owner(rows, trg_fq)
    if len(owners) != 1:
        raise ConfigError(f"{trg_fq} thuộc {owners}: cần đúng 1 pl")
    pl = owners[0]
    if p_pl_name and p_pl_name != pl:
        raise ConfigError(f"{trg_fq} thuộc {pl}, không thuộc {p_pl_name}")
    dag = build_dag(rows, pl)
    edges = dag.nodes[trg_fq]["edges"]
    by_tbl = {}
    for e in edges:
        by_tbl.setdefault(e.src_tbl, []).append(e)
    errs = [f"vai trò '{r}' -> {t}: không có cạnh" for r, t in spec.inputs.items() if t not in by_tbl]
    errs += [f"cạnh từ {e.src_fq} không có vai trò trong spec.inputs" for e in edges if e.src_tbl not in spec.inputs.values()]
    errs += [f"{t} có {len(v)} cạnh (khác schema)" for t, v in by_tbl.items() if len(v) > 1]
    errs += [f"vai trò bắt buộc '{r}' không có trong spec.inputs" for r in spec.required_inputs if r not in spec.inputs]
    if errs:
        raise ConfigError(f"{trg_fq}: spec.inputs lệch config:\n  - " + "\n  - ".join(errs))
    return trg_fq, pl, dag, {r: by_tbl[t][0] for r, t in spec.inputs.items()}


def run_node(spec: NodeSpec, p_exec_id: str = "", p_run_id: str = "", p_pinned_json: str = "", p_mode: str = "RUN",
             p_force: str = "false", p_pl_name: str = "", p_trg_fq: str = "", p_history_limit: str = "1000") -> dict:
    """Chạy 1 node (xem bảng bước ở đầu mục 6).

    Input : spec ; tham số notebook (chuỗi): p_exec_id / p_run_id / p_pinned_json do NB_00 truyền, rỗng = chạy tay ;
            p_mode RUN | DRY_RUN ; p_force true = bỏ qua hộp thư
    Output: dict kết quả (exit value). FAILED / SKIPPED -> ghi log rồi raise RuntimeError
    Ví dụ : run_node(SPEC) chạy tay -> {"status": "SUCCESS", "inserted": 12, "updated": 3, "deactivated": 0, ...}
    """
    started = utc_now()
    mode = (p_mode or "RUN").strip().upper()
    if mode not in NODE_MODES:
        raise ConfigError(f"p_mode '{p_mode}' không hợp lệ, chỉ {NODE_MODES}")
    try:
        pinned = {k: int(v) for k, v in (json.loads(p_pinned_json) if p_pinned_json else {}).items()}
    except Exception as e:
        raise ConfigError(f"p_pinned_json không hợp lệ: {e}") from None
    force = to_bool(p_force)
    history_limit = int(p_history_limit or 1000)
    require_ctrl_columns()
    rows = load_config_rows()
    trg_fq, pl, dag, edges_by_role = _resolve_node(spec, rows, p_pl_name, p_trg_fq)
    edges = dag.nodes[trg_fq]["edges"]
    node = spec.trg_tbl
    exec_id = p_exec_id or str(uuid.uuid4())
    run_id = p_run_id or exec_id
    tag = exec_id.replace("-", "")[:12]
    res = {"status": "FAILED", "node": trg_fq, "pl_name": pl, "mode": mode, "exec_id": exec_id, "error": None,
           "inserted": None, "updated": None, "deactivated": None, "trg_version": None,
           "src_versions": {}, "qg": {"mode": mode, "force": force}}
    own_lock, df = False, None
    log(f"Bắt đầu node {trg_fq} (pl {pl}, mode {mode}, exec {exec_id}, {'qua NB_00' if p_exec_id else 'chạy tay'})",
        node=node)
    try:
        lock = read_flow_lock(pl, LOCK_TIMEOUT_DEFAULT_MIN)
        if p_exec_id:
            if lock is None or lock["lock_exec_id"] != p_exec_id:
                raise ConfigError(f"Khoá luồng {flow_lock_id(pl)} không thuộc exec {p_exec_id} "
                                  f"(đang: {lock['lock_exec_id'] if lock else 'chưa có dòng khoá'})")
        elif mode == "RUN":
            ensure_lock_row(pl, current_notebook_name(dag.nodes[trg_fq]["nb"]))
            ok, who = acquire_flow_lock(pl, exec_id, run_id, LOCK_TIMEOUT_DEFAULT_MIN)
            if not ok:
                raise ConfigError(f"Luồng {pl} đang chạy ({who}); chạy tay lúc này chỉ được p_mode = DRY_RUN")
            own_lock = True
        elif lock is not None and lock["lock_exec_id"] and not lock["expired"]:
            log(f"Luồng đang chạy (exec {lock['lock_exec_id']}): DRY_RUN chỉ đọc, kết quả có thể lệch", "WARN", node)

        if p_exec_id:                                   # cha lỗi / bị bỏ qua trong cùng exec -> bỏ qua
            logs = node_logs(exec_id)
            bad = sorted(p for p in dag.parents[trg_fq] if logs.get(p, {}).get("status") in NODE_BAD)
            if bad:
                res.update(status="SKIPPED", error=f"node cha lỗi / bị bỏ qua: {bad}")
                return res

        states = {e.src_fq: table_state(e.src_fq) for e in edges}
        missing = sorted(fq for fq, s in states.items() if s is None)
        if missing and mode == "DRY_RUN" and all(fq in dag.nodes for fq in missing):
            # DRY_RUN không tạo bảng đích -> node con của node chưa từng RUN không có gì để đọc
            res.update(status="SKIPPED", error=f"DRY_RUN: bảng node cha chưa có (chưa RUN lần nào): {missing}")
            return res
        if missing:
            raise ConfigError(f"Bảng đầu vào chưa có: {missing}")
        read_v = {fq: pinned.get(fq, s["version"]) for fq, s in states.items()}
        bad_pin = sorted(fq for fq, v in read_v.items() if v > states[fq]["version"])
        if bad_pin:
            raise ConfigError(f"Version chốt lớn hơn version hiện tại: {bad_pin}")
        wm = load_edge_wm(edges)
        hist = HistoryCache(DEFAULT_NON_DATA_OPS, history_limit)
        inbox = {}
        for e in edges:
            dirty, reason, vers = edge_status(wm.get((e.src_fq, trg_fq)), states[e.src_fq], read_v[e.src_fq], hist, e.src_fq)
            inbox[e.src_fq] = reason
            res["src_versions"][e.src_fq] = {"v": read_v[e.src_fq], "tid": states[e.src_fq]["table_id"],
                                             "ts": str(states[e.src_fq]["commit_ts"]) if read_v[e.src_fq] == states[e.src_fq]["version"] else None,
                                             "dirty": reason if dirty else None}
        res["qg"]["inbox"] = inbox
        target_missing = not table_exists(trg_fq)
        any_dirty = any(v["dirty"] for v in res["src_versions"].values())
        if not any_dirty and not force and not target_missing:
            res["status"] = "NO_DATA"
            log("Hộp thư rỗng -> NO_DATA", node=node)
            return res
        log(f"New Edge: { {k: v for k, v in inbox.items() if v not in ('UNCHANGED', 'MAINTENANCE_ONLY')} }"
            + (" | p_force" if force else "") + (" | chưa có bảng đích" if target_missing else ""), node=node)

        ctx = NodeContext(spec, trg_fq, edges_by_role, read_v, exec_id, mode)
        empty = [r for r in spec.required_inputs if ctx.is_empty(r)]
        if empty:
            raise QualityGateError(f"Đầu vào bắt buộc rỗng: {[edges_by_role[r].src_fq for r in empty]}")
        df = add_row_hash(spec.build(ctx), spec).persist(StorageLevel.MEMORY_AND_DISK)
        res["qg"].update(quality_gate(spec, df, ctx))
        res["qg"]["metrics"] = ctx.metrics
        diff = diff_counts(df, trg_fq, spec.pk)
        res.update(inserted=diff["inserted"], updated=diff["updated"], deactivated=diff["deactivated"])
        log(f"Diff: insert {diff['inserted']}, update {diff['updated']}, xoá mềm {diff['deactivated']} "
            f"(nguồn {res['qg']['rows']} dòng)", node=node)
        if mode == "RUN":
            changed = ensure_target(trg_fq, df.schema, node)
            if changed:
                log(f"Bảng đích: {changed}", node=node)
            if diff["inserted"] or diff["updated"] or diff["deactivated"]:
                merge_target(trg_fq, df, spec, tag)
            else:
                log("Không có dòng đổi -> bỏ qua MERGE (không tạo commit)", node=node)
            res["trg_version"] = table_state(trg_fq)["version"]
        res["status"] = "SUCCESS"
        return res
    except Exception as e:
        res.update(status="FAILED", error=short_error(e))
        log(res["error"], "ERROR", node)
        return res
    finally:
        ended = utc_now()
        try:
            write_node_log({
                "exec_id": exec_id, "run_id": run_id, "pl_name": pl, "trg_schema": trg_fq.rsplit(".", 1)[0],
                "trg_tbl": node, "priority": dag.level.get(trg_fq), "attempt_no": 1,
                "entity_rows": res["qg"].get("rows"), "inserted_rows": res["inserted"], "updated_rows": res["updated"],
                "deactivated_rows": res["deactivated"], "rejected_rows": 0, "status": res["status"],
                "started_at": started, "ended_at": ended, "duration_ms": int((ended - started).total_seconds() * 1000),
                "error_message": res["error"], "src_versions_json": json_dumps(res["src_versions"]),
                "trg_version": res["trg_version"], "qg_json": json_dumps(res["qg"])[:16000],
            })
        finally:
            if df is not None:
                df.unpersist()
            if own_lock:
                release_flow_lock(pl, exec_id, res["status"])
        log(f"Kết thúc {res['status']} ({(ended - started).total_seconds():.0f}s)", node=node)


def finish_node(res: dict) -> str:
    """Gọi ở cell cuối notebook node: FAILED / SKIPPED -> raise (runMultiple ghi nhận lỗi, node con không chạy);
    còn lại trả chuỗi JSON để notebookutils.notebook.exit."""
    if res["status"] in ("FAILED", "SKIPPED"):
        raise RuntimeError(f"{res['node']} {res['status']}: {res['error']}")
    return json_dumps({k: res[k] for k in ("status", "node", "inserted", "updated", "deactivated", "trg_version")})

## 7. orchestrator — `plan_run`, DAG runMultiple, gom kết quả, tiến cạnh, `orchestrate`

```
[0] kiểm tra tham số (miền giá trị, lock > DAG timeout)       [PLAN: chỉ [3] + in, không khoá, không ghi]
[1] dòng khoá + đóng run RUNNING quá hạn + nhận khoá (không được -> SKIPPED_CONCURRENT, không raise)
[2] run log RUNNING
[3] plan_run: Dirty Edge -> dirty node ∪ p_force_nodes -> + child = parrent; chốt version đầu vào ngoài
    (node của pl khác: version ở run thành công gần nhất của pl đó — lát cắt nhất quán)
[4] runMultiple (concurrency, timeout từng node / cả DAG, retry 0); lỗi runMultiple không dừng NB_00
[5] kết quả từng node = dòng ctrl_log_table_run của exec (node không có dòng = NOT_RUN)
[6] RUN: tiến cạnh của node SUCCESS / NO_DATA tới version node đã đọc; node lỗi: ghi trạng thái, không tiến
[7] đóng run log (+ output_versions_json khi SUCCESS / NO_DATA), nhả khoá; có node lỗi -> raise (pipeline báo lỗi)
```

In [ ]:
# =============================================================================
# 7. orchestrator
# =============================================================================
@dataclass
class Plan:
    dag: Dag
    states: dict                     # fq -> table_state | None (đầu vào + node)
    pinned: dict                     # bảng ngoài -> version chốt
    edges: list                      # [{"edge", "dirty", "reason", "versions", "consumed", "read"}]
    dirty: set
    forced: set
    candidates: list                 # trg_fq theo thứ tự topo


def plan_run(pl_name: str, force_nodes: list, non_data_ops, history_limit: int) -> Plan:
    """Lập kế hoạch 1 lần chạy (chỉ đọc). Bảng ngoài chưa tồn tại -> ConfigError. Node chưa có bảng đích -> Dirty Edge."""
    rows = load_config_rows()
    dag = build_dag(rows, pl_name)
    forced = set()
    for n in force_nodes:
        fq = dag.node_by_name(n)
        if not fq:
            raise ConfigError(f"p_force_nodes: '{n}' không phải node của {pl_name}: {[v['name'] for v in dag.nodes.values()]}")
        forced.add(fq)
    states = {fq: table_state(fq) for fq in sorted({e.src_fq for e in dag.edges} | set(dag.nodes))}
    missing = sorted(fq for fq in dag.external if states[fq] is None)
    if missing:
        raise ConfigError(f"Bảng đầu vào ngoài chưa tồn tại: {missing}")
    pinned, foreign = {}, {}
    for fq, owner in sorted(dag.external.items()):
        v = states[fq]["version"]
        if owner:                                      # node của pl khác: lấy lát cắt nhất quán (§11 chốt 2)
            ov = foreign.setdefault(owner, last_output_versions(owner))
            if fq in ov and int(ov[fq]) <= v:
                v = int(ov[fq])
            else:
                log(f"{fq} (node của {owner}) chưa có output_versions_json -> đọc version hiện tại {v}", "WARN")
        pinned[fq] = v
    wm = load_edge_wm(dag.edges)
    hist = HistoryCache(non_data_ops, history_limit)
    edge_rows = []
    for e in dag.edges:
        st = states.get(e.src_fq)
        consumed = wm.get((e.src_fq, e.trg_fq))
        if e.src_fq in dag.nodes and st is None:
            dirty, reason, vers, rv = True, "PARENT_NOT_BUILT", [], None
        else:
            rv = pinned.get(e.src_fq, st["version"])
            dirty, reason, vers = edge_status(consumed, st, rv, hist, e.src_fq)
        edge_rows.append({"edge": e, "dirty": dirty, "reason": reason, "versions": vers,
                          "consumed": consumed.get("version") if consumed else None, "read": rv})
    dirty = {r["edge"].trg_fq for r in edge_rows if r["dirty"]} | {fq for fq in dag.nodes if states[fq] is None}
    start = dirty | forced
    cand = start | dag.descendants(start)
    return Plan(dag, states, pinned, edge_rows, dirty, forced, [fq for fq in dag.order if fq in cand])


def plan_summary(plan: Plan) -> dict:
    """Tóm tắt kế hoạch (in + ghi read_note)."""
    return {
        "dirty_edges": [f"{r['edge'].trg_tbl} <- {r['edge'].src_fq}: {r['reason']} "
                        f"(đã đọc {r['consumed']}, đọc {r['read']}{', commit ' + str(r['versions']) if r['versions'] else ''})"
                        for r in plan.edges if r["dirty"]],
        "dirty_nodes": sorted(plan.dag.nodes[fq]["name"] for fq in plan.dirty),
        "forced": sorted(plan.dag.nodes[fq]["name"] for fq in plan.forced),
        "candidates": [f"L{plan.dag.level[fq]} {plan.dag.nodes[fq]['name']} ({plan.dag.nodes[fq]['nb']})"
                       for fq in plan.candidates],
        "pinned": plan.pinned,
    }


def print_plan(plan: Plan) -> None:
    s = plan_summary(plan)
    log(f"Kế hoạch {plan.dag.pl_name}: {len(plan.dag.nodes)} node, {len(plan.dag.edges)} cạnh, "
        f"{len(s['dirty_edges'])} Dirty Edge, {len(plan.candidates)} node ứng viên")
    for line in s["dirty_edges"]:
        log(f"  Dirty Edge: {line}")
    for line in s["candidates"]:
        log(f"  chạy: {line}")
    if s["forced"]:
        log(f"  ép chạy: {s['forced']}")


def build_run_multiple_dag(plan: Plan, exec_id: str, run_id: str, node_mode: str, node_timeout_min: int,
                           dag_timeout_min: int, max_parallel: int, history_limit: int) -> dict:
    """DAG cho notebookutils.notebook.runMultiple: 1 activity / node ứng viên, phụ thuộc = node cha cũng là ứng viên.
    args chỉ chứa version chốt của đầu vào ngoài của chính node. [VERIFY] tên trường theo tài liệu runMultiple."""
    cand = set(plan.candidates)
    acts = []
    for fq in plan.candidates:
        n = plan.dag.nodes[fq]
        srcs = {e.src_fq for e in n["edges"]}
        acts.append({
            "name": n["name"],
            "path": n["nb"],
            "timeoutPerCellInSeconds": int(node_timeout_min) * 60,
            "retry": 0,
            "args": {"p_exec_id": exec_id, "p_run_id": run_id, "p_pl_name": plan.dag.pl_name, "p_trg_fq": fq,
                     "p_mode": node_mode, "p_force": "true" if fq in plan.forced else "false",
                     "p_history_limit": str(history_limit),
                     "p_pinned_json": json_dumps({k: v for k, v in plan.pinned.items() if k in srcs})},
            "dependencies": [plan.dag.nodes[p]["name"] for p in sorted(plan.dag.parents[fq]) if p in cand],
        })
    return {"activities": acts, "timeoutInSeconds": int(dag_timeout_min) * 60, "concurrency": int(max_parallel)}


def collect_outcomes(exec_id: str, plan: Plan) -> dict:
    """Kết quả từng node ứng viên từ ctrl_log_table_run (không phụ thuộc dạng trả về của runMultiple).
    Output: {trg_fq: {"status", "error", "src_versions" (dict), "log" (dict | None)}}"""
    logs = node_logs(exec_id)
    out = {}
    for fq in plan.candidates:
        r = logs.get(fq)
        if r is None:
            out[fq] = {"status": "NOT_RUN", "error": "không có dòng log (không khởi động / timeout / bị huỷ)",
                       "src_versions": {}, "log": None}
            continue
        sv = json.loads(r["src_versions_json"]) if r.get("src_versions_json") else {}
        out[fq] = {"status": r["status"], "error": r.get("error_message"), "src_versions": sv, "log": r}
    return out


def advance_edges(plan: Plan, outcomes: dict, run_id: str, tag: str) -> int:
    """Tiến cạnh của node SUCCESS / NO_DATA tới version node đã đọc (src_versions_json của chính node);
    node lỗi: chỉ ghi trạng thái + lỗi lên cạnh. 1 MERGE. Output: số cạnh được tiến."""
    rows, n_adv = [], 0
    for fq in plan.candidates:
        n, o = plan.dag.nodes[fq], outcomes[fq]
        for e in n["edges"]:
            base = {"watermark_id": e.wm_id, "src_schema": e.src_schema, "src_tbl": e.src_tbl,
                    "trg_schema": e.trg_schema, "trg_tbl": e.trg_tbl, "flow_name": n["nb"], "last_run_id": run_id}
            sv = o["src_versions"].get(e.src_fq)
            if o["status"] in NODE_OK and sv and sv.get("v") is not None:
                ts = sv.get("ts")
                rows.append({**base, "advance": True, "last_src_version": int(sv["v"]), "last_src_table_id": sv.get("tid"),
                             "watermark_value": datetime.fromisoformat(ts) if ts and ts != "None" else None,
                             "status": "SUCCESS"})
                n_adv += 1
            else:
                st = o["status"] if o["status"] in NODE_BAD else "FAILED"
                rows.append({**base, "advance": False, "status": st, "error_message": (o["error"] or st)[:1000]})
    merge_edge_wm(rows, tag)
    return n_adv


def _validate_orch_params(p: dict) -> dict:
    """Kiểm tra miền giá trị tham số NB_00 trước khi làm gì (bài học C2). Sai -> ConfigError liệt kê hết."""
    errs, out = [], {}
    out["pl_name"] = (p.get("p_pl_name") or "").strip()
    if not is_identifier(out["pl_name"]):
        errs.append(f"p_pl_name '{out['pl_name']}' không hợp lệ")
    out["mode"] = (p.get("p_mode") or "RUN").strip().upper()
    if out["mode"] not in RUN_MODES:
        errs.append(f"p_mode '{p.get('p_mode')}' chỉ {RUN_MODES}")
    for k, lo, hi in (("p_max_parallel", 1, 8), ("p_node_timeout_min", 1, 240), ("p_dag_timeout_min", 1, 720),
                      ("p_lock_timeout_min", 5, 1440), ("p_history_limit", 10, 100000)):
        try:
            v = int(str(p.get(k)).strip())
            if not lo <= v <= hi:
                raise ValueError
            out[k] = v
        except (TypeError, ValueError):
            errs.append(f"{k} = '{p.get(k)}' phải là số nguyên trong [{lo}, {hi}]")
    if not errs:
        if out["p_node_timeout_min"] > out["p_dag_timeout_min"]:
            errs.append("p_node_timeout_min phải <= p_dag_timeout_min")
        if out["p_lock_timeout_min"] <= out["p_dag_timeout_min"] + 5:
            errs.append("p_lock_timeout_min phải > p_dag_timeout_min + 5 (khoá không được hết hạn khi DAG còn chạy)")
    out["force_nodes"] = split_csv(p.get("p_force_nodes"))
    ops = split_csv(p.get("p_non_data_ops"))
    out["non_data_ops"] = tuple(o.upper() for o in ops) if ops else DEFAULT_NON_DATA_OPS
    out["run_id"] = (p.get("p_run_id") or "").strip()
    if errs:
        raise ConfigError("Tham số NB_00 sai:\n  - " + "\n  - ".join(errs))
    return out


def orchestrate(**params) -> dict:
    """Thân NB_00 (xem sơ đồ đầu mục 7). params = các tham số notebook p_*.

    Output: dict tóm tắt. Node lỗi / NB_00 lỗi -> raise sau khi đã đóng run log và nhả khoá.
    Ví dụ : orchestrate(p_pl_name="PL_VV_TRANSFORM_SLV_TO_GLD_1H", p_mode="PLAN", ...) -> in kế hoạch, không ghi
    """
    p = _validate_orch_params(params)
    pl, mode = p["pl_name"], p["mode"]
    exec_id = str(uuid.uuid4())
    run_id = p["run_id"] or exec_id
    nb_name = current_notebook_name("NB_00_ORCHES_SLV_TO_GLD")
    started = utc_now()
    require_ctrl_columns()
    log(f"NB_00 {pl} mode {mode} exec {exec_id} run {run_id}")
    if mode == "PLAN":
        plan = plan_run(pl, p["force_nodes"], p["non_data_ops"], p["p_history_limit"])
        print_plan(plan)
        dag_json = build_run_multiple_dag(plan, "<exec_id>", run_id, "RUN", p["p_node_timeout_min"],
                                          p["p_dag_timeout_min"], p["p_max_parallel"], p["p_history_limit"])
        return {"status": "PLAN", "plan": plan_summary(plan), "run_multiple_dag": dag_json}

    lock_id = ensure_lock_row(pl, nb_name)
    base = {"exec_id": exec_id, "run_id": run_id, "pl_name": pl, "nb_name": nb_name, "watermark_id": lock_id,
            "run_mode": mode, "table_filter": ",".join(p["force_nodes"]) or None, "started_at": started,
            "read_mode": "VERSION_INBOX", "run_params": json_dumps(params)}
    close_stale_runs(pl, exec_id, p["p_lock_timeout_min"])
    ok, who = acquire_flow_lock(pl, exec_id, run_id, p["p_lock_timeout_min"])
    if not ok:
        write_run_log({**base, "status": "SKIPPED_CONCURRENT", "ended_at": utc_now(), "error_message": who})
        log(f"Bỏ qua: {who}", "WARN")
        return {"status": "SKIPPED_CONCURRENT", "reason": who}

    status, err, out_versions, read_note = "FAILED", None, None, None
    counts = {"tbl_total_count": 0, "tbl_success_count": 0, "tbl_failed_count": 0, "tbl_no_data_count": 0}
    summary = {"exec_id": exec_id, "pl_name": pl, "mode": mode}
    try:
        write_run_log({**base, "status": "RUNNING"})
        plan = plan_run(pl, p["force_nodes"], p["non_data_ops"], p["p_history_limit"])
        print_plan(plan)
        read_note = json_dumps(plan_summary(plan))[:8000]
        if not plan.candidates:
            status = "NO_DATA"
            if mode == "RUN":
                out_versions = {fq: plan.states[fq]["version"] for fq in plan.dag.nodes}
        else:
            dag_json = build_run_multiple_dag(plan, exec_id, run_id, mode, p["p_node_timeout_min"],
                                              p["p_dag_timeout_min"], p["p_max_parallel"], p["p_history_limit"])
            rm_err = None
            try:
                notebookutils.notebook.runMultiple(dag_json, {"displayDAGViaGraphviz": False})   # noqa: F821
            except Exception as e:                       # node lỗi -> runMultiple raise; kết quả thật đọc từ log
                rm_err = short_error(e, 1500)
                log(f"runMultiple báo lỗi: {rm_err}", "WARN")
            outcomes = collect_outcomes(exec_id, plan)
            for fq, o in outcomes.items():
                log(f"  {plan.dag.nodes[fq]['name']}: {o['status']}" + (f" — {o['error']}" if o["error"] else ""))
            if mode == "RUN":
                n_adv = advance_edges(plan, outcomes, run_id, exec_id.replace("-", "")[:12])
                log(f"Tiến {n_adv} cạnh")
            bad = {plan.dag.nodes[fq]["name"]: o["status"] for fq, o in outcomes.items() if o["status"] in NODE_BAD}
            counts = {"tbl_total_count": len(outcomes),
                      "tbl_success_count": sum(o["status"] == "SUCCESS" for o in outcomes.values()),
                      "tbl_failed_count": len(bad),
                      "tbl_no_data_count": sum(o["status"] == "NO_DATA" for o in outcomes.values())}
            status = "PARTIAL_FAILED" if bad else "SUCCESS"
            err = (f"node lỗi: {json_dumps(bad)}" + (f" | runMultiple: {rm_err}" if rm_err else "")) if bad else None
            if status == "SUCCESS" and mode == "RUN":
                out_versions = {fq: (table_state(fq) or {}).get("version") for fq in plan.dag.nodes}
            summary["outcomes"] = {plan.dag.nodes[fq]["name"]: o["status"] for fq, o in outcomes.items()}
    except Exception as e:
        status, err = "FAILED", short_error(e)
        raise
    finally:
        ended = utc_now()
        try:
            update_run_log(exec_id, {"status": status, "ended_at": ended,
                                     "duration_ms": int((ended - started).total_seconds() * 1000),
                                     "error_message": err, "read_note": read_note,
                                     "output_versions_json": json_dumps(out_versions) if out_versions else None,
                                     **counts})
        finally:
            release_flow_lock(pl, exec_id, status)
        log(f"NB_00 kết thúc {status} ({(ended - started).total_seconds():.0f}s)")
    summary.update(status=status, error=err, **counts)
    if status not in ("SUCCESS", "NO_DATA"):
        raise RuntimeError(f"{pl} {status}: {err}")
    return summary

## 8. build helpers — chuẩn hoá chữ, bỏ dấu, slug, uuid md5, epoch, JSON

| Hàm | Ví dụ |
|---|---|
| `clean_str(c)` | `"  "` / `"__debezium_unavailable_value"` -> NULL ; `" a "` -> `"a"` |
| `norm_text(c)` | `"  Phú   Quốc "` -> `"phú quốc"` (rỗng -> NULL) |
| `norm_admin(c)` | `"Phường Bến Nghé"` -> `"bến nghé"` ; `"TP. Hồ Chí Minh"` -> `"hồ chí minh"` (tiền tố như `normalize_city_name` của 2.1) |
| `unaccent(c)` | `"Phở Đức"` -> `"Pho Duc"` (giữ hoa / thường) |
| `slugify(c)` | `"Phở Hà Nội #1"` -> `"pho-ha-noi-1"` |
| `uuid_md5(*cols)` | md5(nối chuỗi) dạng 8-4-4-4-12 (= `HASH_MD5_UUID` của extract) |
| `epoch_to_ts(c)` | `1787048668` / `1787048668000` -> timestamp (tự nhận giây / mili giây như 2.1) |
| `json_scalar(c, path)` | Giá trị tại path; là mảng -> phần tử đầu (như `build_row_base` 2.1) |
| `json_sorted_array(c)` | Mảng -> JSON đã bỏ trùng + sắp xếp (băm ổn định) |

In [ ]:
# =============================================================================
# 8. build helpers
# =============================================================================
TOAST_SENTINEL = "__debezium_unavailable_value"
ADMIN_PREFIX_PATTERNS = (r"^(thành phố|tỉnh|tp\.?)\s+", r"^(phường|xã|thị trấn|tt\.?)\s+")   # 2.1 normalize_city_name


def _build_unaccent_map() -> tuple:
    """Bảng chuyển chữ Latin có dấu -> chữ gốc (sinh từ Unicode NFD, gồm đủ chữ Việt) + đ/Đ."""
    src, dst = [], []
    for cp in list(range(0x00C0, 0x0250)) + list(range(0x1E00, 0x1F00)):
        ch = chr(cp)
        base = unicodedata.normalize("NFD", ch)[0]
        if base != ch and base.isascii() and base.isalpha():
            src.append(ch)
            dst.append(base)
    return "".join(src) + "đĐ", "".join(dst) + "dD"


UNACCENT_SRC, UNACCENT_DST = _build_unaccent_map()


def clean_str(c) -> Column:
    c = F.trim(F.col(c) if isinstance(c, str) else c)
    return F.when((c == "") | (c == TOAST_SENTINEL), F.lit(None).cast("string")).otherwise(c)


def norm_text(c) -> Column:
    c = F.lower(F.trim(F.regexp_replace(F.col(c) if isinstance(c, str) else c, r"\s+", " ")))
    return F.when(c == "", F.lit(None).cast("string")).otherwise(c)


def norm_admin(c) -> Column:
    c = norm_text(c)
    for pat in ADMIN_PREFIX_PATTERNS:
        c = F.regexp_replace(c, pat, "")
    c = F.trim(c)
    return F.when(c == "", F.lit(None).cast("string")).otherwise(c)


def unaccent(c) -> Column:
    c = F.col(c) if isinstance(c, str) else c
    return F.regexp_replace(F.translate(c, UNACCENT_SRC, UNACCENT_DST), r"\p{M}+", "")


def slugify(c) -> Column:
    s = F.lower(unaccent(c))
    s = F.regexp_replace(s, r"[^a-z0-9]+", "-")
    return F.regexp_replace(s, r"^-+|-+$", "")


def uuid_md5(*cols) -> Column:
    parts = [F.col(c) if isinstance(c, str) else c for c in cols]
    return F.regexp_replace(F.md5(F.concat(*parts)), r"^(.{8})(.{4})(.{4})(.{4})(.{12})$", "$1-$2-$3-$4-$5")


def epoch_to_ts(c) -> Column:
    c = (F.col(c) if isinstance(c, str) else c).cast("long")
    return (F.when(c.isNull() | (c == 0), F.lit(None).cast("timestamp"))
             .when(c > 9999999999, F.timestamp_seconds((c / 1000).cast("long")))
             .otherwise(F.timestamp_seconds(c)))


def json_scalar(json_col: str, path_col: str) -> Column:
    """get_json_object(json_col, path_col) với path là CỘT (mỗi dòng 1 path, vd lấy từ ref); giá trị là mảng JSON
    -> phần tử đầu (như 2.1); rỗng / sentinel -> NULL. Ví dụ: amentities, '$.facilities.main_cuisine' -> 'Vietnamese'."""
    raw = F.expr(f"get_json_object(`{json_col}`, `{path_col}`)")
    return clean_str(F.when(F.trim(raw).startswith("["), F.get_json_object(raw, "$[0]")).otherwise(raw))


def json_array_at(json_col: str, path_col: str) -> Column:
    """Mảng chuỗi tại path (path là cột); giá trị đơn -> mảng 1 phần tử; không có -> NULL."""
    raw = F.expr(f"get_json_object(`{json_col}`, `{path_col}`)")
    return (F.when(raw.isNull() | (F.trim(raw) == ""), F.lit(None).cast("array<string>"))
             .when(F.trim(raw).startswith("["), F.from_json(raw, "array<string>"))
             .otherwise(F.array(raw)))


def json_sorted_array(c) -> Column:
    """Mảng chuỗi -> JSON mảng đã bỏ NULL / trùng và sắp xếp; mảng rỗng / NULL -> NULL."""
    c = F.col(c) if isinstance(c, str) else c
    a = F.array_sort(F.array_distinct(F.filter(c, lambda x: x.isNotNull() & (F.trim(x) != ""))))
    return F.when(F.size(a) > 0, F.to_json(a))